# Bus Corridor Simulator

Model a one-way urban bus service and study how changes to departures affect passenger waiting. This first notebook verifies a single stop in normal service, using fixed times that can be calculated by hand.

## 1. Problem and question

Buses reaching a stop close together sometimes wait for the same boarding space. We will measure what this means for passengers on a straight corridor and compare departure policies after the baseline is verified.

**Planned road:** Origin → Stop 1 → Stop 2 → Terminal. This checkpoint isolates **Stop 1**.

## 2. Model, assumptions and equations

Two buses arrive at a single berth. If it is busy, the next bus queues in arrival order. Passengers who have reached the stop can board when service starts. This case uses fixed arrivals and dwell times so every event can be checked.

$$W_{\mathrm{bus}}=t_{\mathrm{service\ start}}-t_{\mathrm{bus\ arrival}},\qquad
W_{\mathrm{passenger}}=t_{\mathrm{boarding}}-t_{\mathrm{passenger\ arrival}}.$$

All times are in seconds. The model applies next-event time advance and FIFO resource queueing. The project theory file records the input modelling and Monte Carlo methods planned for later increments.

## 3. Implementation

The calendar sorts by time, then event priority, then insertion order. A berth release and any arrivals at a given time happen before the next service starts. An earlier bus already in the queue retains its place.

In [1]:
from collections import deque
from heapq import heappop, heappush
from itertools import count

# Tuples contain (bus ID, arrival time in seconds, dwell in seconds).
BUS_INPUTS = (("A", 70, 30), ("B", 80, 10))
PASSENGER_ARRIVALS = (85,)

EVENT_PRIORITY = {
    "berth_release": 0,
    "passenger_arrival": 1,
    "bus_arrival": 2,
    "service_start": 3,
}

In [2]:
def simulate_stop(bus_inputs, passenger_arrivals):
    calendar = []
    sequence = count()
    trace = []
    bus_queue = deque()
    passenger_queue = deque()
    berth_bus = None
    bus_times = {}
    dwell_times = {}
    passenger_waits = {}
    last_time = 0

    def schedule(time, event, entity):
        heappush(calendar, (time, EVENT_PRIORITY[event], next(sequence), event, entity))

    for bus, arrival, dwell in bus_inputs:
        bus_times[bus] = {"arrival": arrival}
        dwell_times[bus] = dwell
        schedule(arrival, "bus_arrival", bus)

    for passenger, arrival in enumerate(passenger_arrivals, start=1):
        schedule(arrival, "passenger_arrival", passenger)

    while calendar:
        now, _, _, event, entity = heappop(calendar)
        assert now >= last_time
        last_time = now
        trace.append((now, event, entity))

        if event == "passenger_arrival":
            passenger_queue.append((entity, now))

        elif event == "bus_arrival":
            bus_queue.append(entity)
            if berth_bus is None and len(bus_queue) == 1:
                schedule(now, "service_start", entity)

        elif event == "service_start":
            assert berth_bus is None, "The berth can serve one bus at a time."
            first_bus = bus_queue.popleft()
            assert first_bus == entity, "Waiting buses must be served FIFO."
            berth_bus = entity
            bus_times[entity]["service_start"] = now
            bus_times[entity]["berth_wait"] = now - bus_times[entity]["arrival"]
            while passenger_queue:
                passenger, arrival = passenger_queue.popleft()
                passenger_waits[passenger] = now - arrival
                trace.append((now, f"passenger {passenger} boards", entity))
            schedule(now + dwell_times[entity], "berth_release", entity)

        elif event == "berth_release":
            assert berth_bus == entity
            bus_times[entity]["departure"] = now
            berth_bus = None
            if bus_queue:
                schedule(now, "service_start", bus_queue[0])

    assert len(passenger_waits) + len(passenger_queue) == len(passenger_arrivals)
    assert all("departure" in timings for timings in bus_times.values())
    return trace, bus_times, passenger_waits

## 4. Verification

| Quantity | Expected |
|---|---:|
| Bus A departs | 100 s |
| Bus B starts boarding | 100 s |
| Bus B berth wait | 20 s |
| Passenger platform wait | 15 s |

The second case adds a bus and a passenger at exactly 100 s to check ordering.

In [3]:
trace, buses, passenger_waits = simulate_stop(BUS_INPUTS, PASSENGER_ARRIVALS)
print("Time (s)  Event                       Entity")
for time, event, entity in trace:
    print(f"{time:8g}  {event:27s} {entity}")

assert buses["A"]["departure"] == 100
assert buses["B"]["service_start"] == 100
assert buses["B"]["berth_wait"] == 20
assert passenger_waits[1] == 15
print("PASS: Bus B waited 20 s for the berth. The passenger waited 15 s.")

tie_trace, tie_buses, tie_waits = simulate_stop(
    BUS_INPUTS + (("C", 100, 10),), PASSENGER_ARRIVALS + (100,)
)
assert [event for time, event, _ in tie_trace if time == 100] == [
    "berth_release", "passenger_arrival", "bus_arrival", "service_start",
    "passenger 1 boards", "passenger 2 boards",
]
assert tie_buses["B"]["service_start"] == 100
assert tie_buses["C"]["service_start"] == 110
assert tie_waits[2] == 0
print("PASS: FIFO is preserved at a tie, and a passenger arriving at service start can board.")

Time (s)  Event                       Entity
      70  bus_arrival                 A
      70  service_start               A
      80  bus_arrival                 B
      85  passenger_arrival           1
     100  berth_release               A
     100  service_start               B
     100  passenger 1 boards          B
     110  berth_release               B
PASS: Bus B waited 20 s for the berth. The passenger waited 15 s.
PASS: FIFO is preserved at a tie, and a passenger arriving at service start can board.


## 5. Experiments and results

The trace reproduces the hand calculation. This is a verification run with fixed inputs. Policy comparisons will begin once buses and passengers can complete journeys along the full corridor.

## 6. Discussion and limitations

The starter separates bus berth waiting from passenger waiting. It has no inter-stop travel or destination tracking yet. Next, connect the stops using fixed link times and follow one passenger from origin to destination.

## 7. References

- [Jayasinghe et al., Galle Road context](https://doi.org/10.4038/jsalt.v3i1.57)
- [Herath et al., boarding and alighting dwell model](https://doi.org/10.11175/easts.15.2109)
- [Shen et al., queues and headway control in a busy corridor](https://doi.org/10.1016/j.tra.2025.104752)